# Sistema Inteligente de Extracción Automática de Información Académica
## Proyecto PC3 – Procesamiento de Lenguaje Natural

| | |
|---|---|
| **Curso** | Ciencia de Datos |
| **Docente** | Mg. Aldo Meza |
| **Fecha** | Mayo 2026 |

**Integrantes:**
- Geronimo Ccoillor, Fernando
- Morales Morales, Flavio Oscar
- Nolasco Palomino, Josselyn Nayeli
- Vargas Estela, Kopeycor
- Yucra Paredes, Carmen Rosa

---

### Descripción del Proyecto

Este notebook implementa un sistema completo de PLN orientado a procesar documentos académicos en PDF y extraer automáticamente información relevante. El documento de referencia utilizado es un artículo científico sobre **deserción universitaria** en el contexto peruano.

### Técnicas de PLN implementadas

| # | Técnica | Librería |
|---|---------|----------|
| 1 | Reconocimiento de Entidades Nombradas (NER) | spaCy |
| 2 | Extracción de palabras clave (TF-IDF) | scikit-learn |
| 3 | Análisis de Sentimiento | TextBlob |
| 4 | Nube de Palabras (WordCloud) | wordcloud |
| 5 | Topic Modeling (LDA) | scikit-learn |
| 6 | Extracción de secciones académicas | NLTK + Regex |

### Flujo del Proyecto

1. Instalación y configuración del entorno
2. Extracción de texto desde PDF
3. Preprocesamiento textual
4. Reconocimiento de entidades nombradas (NER)
5. Extracción de términos clave con TF-IDF
6. Análisis de sentimiento
7. Nube de palabras
8. Topic Modeling (LDA)
9. Extracción de secciones académicas
10. Consolidación y exportación de resultados

## Fase 1. Instalación de librerías

In [ ]:
# Instalación de todas las dependencias necesarias
!pip install pdfplumber spacy nltk scikit-learn textblob wordcloud matplotlib pandas numpy streamlit -q

In [ ]:
# Descarga del modelo de spaCy en español
!python -m spacy download es_core_news_sm -q

In [ ]:
# Descarga de recursos NLTK
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
print("Recursos descargados correctamente")

## Fase 2. Carga de librerías y configuración

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
from wordcloud import WordCloud
import pdfplumber
import warnings
warnings.filterwarnings('ignore')

# Cargar modelo spaCy en español
nlp = spacy.load("es_core_news_sm")

# Stopwords en español
stop_words = set(stopwords.words('spanish'))

print("✓ Librerías cargadas correctamente")
print(f"✓ spaCy versión: {spacy.__version__}")

## Fase 3. Extracción de texto desde PDF

In [ ]:
def extraer_texto_pdf(ruta_pdf):
    """
    Extrae texto página por página desde un archivo PDF usando pdfplumber.
    
    Args:
        ruta_pdf (str): Ruta al archivo PDF.
    Returns:
        str: Texto completo del documento.
    """
    texto_total = ""
    with pdfplumber.open(ruta_pdf) as pdf:
        for i, pagina in enumerate(pdf.pages):
            texto = pagina.extract_text()
            if texto:
                texto_total += f"\n--- Página {i+1} ---\n{texto}\n"
    return texto_total

print("✓ Función definida: extraer_texto_pdf")

In [ ]:
# Ruta del documento académico
# Si usas Google Colab: sube el PDF y ajusta la ruta
RUTA_PDF = "Dialnet-DesercionUniversitaria.pdf"

texto_documento = extraer_texto_pdf(RUTA_PDF)

print(f"✓ Texto extraído: {len(texto_documento):,} caracteres")
print(f"✓ Páginas procesadas: {texto_documento.count('--- Página')}")
print("\n--- Vista previa (primeros 1000 caracteres) ---")
print(texto_documento[:1000])

## Fase 4. Preprocesamiento textual

Se aplican las siguientes técnicas de limpieza:
- Conversión a minúsculas
- Eliminación de números y caracteres especiales  
- Eliminación de stopwords
- Lematización con spaCy

In [ ]:
def limpiar_texto(texto):
    """
    Limpia y normaliza el texto aplicando:
    - Conversión a minúsculas
    - Eliminación de saltos de línea, números y caracteres especiales
    - Eliminación de stopwords
    - Lematización (forma base de cada palabra)
    
    Args:
        texto (str): Texto crudo.
    Returns:
        str: Texto preprocesado.
    """
    # Paso 1: minúsculas
    texto = texto.lower()
    
    # Paso 2: eliminar saltos de línea
    texto = texto.replace("\n", " ")
    
    # Paso 3: eliminar números
    texto = re.sub(r'\d+', '', texto)
    
    # Paso 4: eliminar caracteres especiales (mantener solo letras y espacios)
    texto = re.sub(r'[^\w\s]', '', texto)
    
    # Paso 5: tokenizar, eliminar stopwords y lematizar con spaCy
    doc = nlp(texto)
    tokens = [
        token.lemma_
        for token in doc
        if token.text not in stop_words
        and not token.is_space
        and len(token.text) > 2
    ]
    
    return " ".join(tokens)

print("✓ Función definida: limpiar_texto")

In [ ]:
# Aplicar preprocesamiento
texto_limpio = limpiar_texto(texto_documento)

print(f"✓ Texto limpio: {len(texto_limpio):,} caracteres")
print("\n--- Vista previa del texto preprocesado ---")
print(texto_limpio[:800])

## Fase 5. Reconocimiento de Entidades Nombradas (NER)

SpaCy identifica automáticamente:
- **PER** → Personas / posibles autores
- **ORG** → Organizaciones / instituciones
- **LOC** → Lugares geográficos
- **MISC** → Entidades misceláneas

In [ ]:
def extraer_entidades(texto):
    """
    Aplica el modelo NER de spaCy para detectar entidades nombradas.
    
    Args:
        texto (str): Texto original (sin limpiar, para preservar mayúsculas).
    Returns:
        pd.DataFrame: Entidades con columnas [Entidad, Tipo, Inicio, Fin].
    """
    doc = nlp(texto)
    entidades = [
        {"Entidad": ent.text, "Tipo": ent.label_,
         "Inicio": ent.start_char, "Fin": ent.end_char}
        for ent in doc.ents
    ]
    return pd.DataFrame(entidades)

print("✓ Función definida: extraer_entidades")

In [ ]:
# Aplicar NER al texto original
df_entidades = extraer_entidades(texto_documento)

print(f"✓ Total de entidades detectadas: {len(df_entidades)}")
print("\n--- Distribución por tipo ---")
print(df_entidades['Tipo'].value_counts())

In [ ]:
# Entidades únicas ordenadas
df_entidades_unicas = (
    df_entidades
    .drop_duplicates(subset=['Entidad', 'Tipo'])
    .sort_values(['Tipo', 'Entidad'])
    .reset_index(drop=True)
)

print(f"Entidades únicas: {len(df_entidades_unicas)}")
df_entidades_unicas.head(20)

In [ ]:
# Frecuencia de entidades
df_freq_entidades = (
    df_entidades
    .groupby(['Entidad', 'Tipo'])
    .size()
    .reset_index(name='Frecuencia')
    .sort_values('Frecuencia', ascending=False)
)

print("Top 15 entidades más frecuentes:")
df_freq_entidades.head(15)

In [ ]:
# Visualización: distribución de tipos de entidades
conteo_tipos = df_entidades_unicas['Tipo'].value_counts()

plt.figure(figsize=(7, 4))
bars = plt.bar(conteo_tipos.index, conteo_tipos.values,
               color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.title('Distribución de Entidades Nombradas por Tipo', fontsize=13)
plt.xlabel('Tipo de Entidad')
plt.ylabel('Cantidad')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(int(bar.get_height())), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Personas detectadas (posibles autores)
autores = df_entidades[df_entidades['Tipo'] == 'PER']['Entidad'].drop_duplicates().tolist()
print("Posibles autores/personas mencionadas:")
print(autores[:15])

# Organizaciones detectadas
instituciones = df_entidades[df_entidades['Tipo'] == 'ORG']['Entidad'].drop_duplicates().tolist()
print("\nInstituciones u organizaciones:")
print(instituciones[:15])

## Fase 6. Extracción de palabras clave con TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) asigna mayor peso a los términos que son frecuentes en el documento pero poco comunes en general, identificando los conceptos más representativos.

In [ ]:
def extraer_keywords_tfidf(texto, n=25):
    """
    Extrae los n términos más relevantes del documento usando TF-IDF.
    Incluye unigramas y bigramas (pares de palabras).
    
    Args:
        texto (str): Texto del documento.
        n (int): Número de términos a retornar.
    Returns:
        pd.DataFrame: Términos con su puntaje TF-IDF.
    """
    vectorizer = TfidfVectorizer(
        max_features=1500,
        stop_words=list(stop_words),
        ngram_range=(1, 2)   # unigramas y bigramas
    )
    matriz = vectorizer.fit_transform([texto])
    palabras = vectorizer.get_feature_names_out()
    puntajes = matriz.toarray()[0]
    
    df_keywords = pd.DataFrame({
        'Término': palabras,
        'Puntaje TF-IDF': puntajes
    })
    return df_keywords.sort_values('Puntaje TF-IDF', ascending=False).head(n).reset_index(drop=True)

print("✓ Función definida: extraer_keywords_tfidf")

In [ ]:
# Extraer top 25 keywords
df_keywords = extraer_keywords_tfidf(texto_documento, n=25)
print("Top 25 términos TF-IDF:")
df_keywords

In [ ]:
# Visualización TF-IDF
plt.figure(figsize=(10, 7))
plt.barh(df_keywords['Término'][:20], df_keywords['Puntaje TF-IDF'][:20], color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('Puntaje TF-IDF', fontsize=12)
plt.ylabel('Término', fontsize=12)
plt.title('Top 20 Términos más Relevantes (TF-IDF)', fontsize=14)
plt.tight_layout()
plt.show()

## Fase 7. Análisis de Sentimiento

**TextBlob** evalúa cada segmento del texto y calcula:
- **Polaridad**: -1 (muy negativo) → +1 (muy positivo)
- **Subjetividad**: 0 (objetivo) → 1 (subjetivo)

Esto permite entender el tono emocional del documento académico.

In [ ]:
def analisis_sentimiento(texto, max_segmentos=30):
    """
    Analiza el sentimiento de cada segmento/párrafo del documento.
    
    Args:
        texto (str): Texto completo.
        max_segmentos (int): Número máximo de párrafos a analizar.
    Returns:
        pd.DataFrame: Polaridad, subjetividad y etiqueta por segmento.
    """
    parrafos = [p.strip() for p in texto.split("\n") if len(p.strip()) > 60]
    resultados = []
    
    for p in parrafos[:max_segmentos]:
        blob = TextBlob(p)
        pol  = blob.sentiment.polarity
        subj = blob.sentiment.subjectivity
        etiqueta = "Positivo" if pol > 0.05 else ("Negativo" if pol < -0.05 else "Neutro")
        resultados.append({
            'Segmento': p[:120] + '...',
            'Polaridad': round(pol, 4),
            'Subjetividad': round(subj, 4),
            'Sentimiento': etiqueta
        })
    return pd.DataFrame(resultados)

print("✓ Función definida: analisis_sentimiento")

In [ ]:
# Aplicar análisis de sentimiento
df_sentimiento = analisis_sentimiento(texto_documento)

# Resumen
print("Distribución de sentimientos:")
print(df_sentimiento['Sentimiento'].value_counts())
print(f"\nPolaridad promedio: {df_sentimiento['Polaridad'].mean():.4f}")
print(f"Subjetividad promedio: {df_sentimiento['Subjetividad'].mean():.4f}")

In [ ]:
# Vista de los primeros resultados
df_sentimiento.head(10)

In [ ]:
# Visualización: polaridad por segmento
colores = ['green' if s == 'Positivo' else ('red' if s == 'Negativo' else 'gray')
           for s in df_sentimiento['Sentimiento']]

plt.figure(figsize=(12, 4))
plt.bar(range(len(df_sentimiento)), df_sentimiento['Polaridad'], color=colores)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('Segmento #', fontsize=12)
plt.ylabel('Polaridad', fontsize=12)
plt.title('Análisis de Sentimiento por Segmento del Documento', fontsize=14)
plt.tight_layout()
plt.show()

# Gráfico de distribución de sentimientos
dist = df_sentimiento['Sentimiento'].value_counts()
plt.figure(figsize=(5, 4))
plt.pie(dist.values, labels=dist.index,
        colors=['green', 'gray', 'red'][:len(dist)],
        autopct='%1.1f%%', startangle=90)
plt.title('Distribución de Sentimientos', fontsize=13)
plt.show()

## Fase 8. Nube de Palabras (WordCloud)

Visualiza gráficamente los términos más frecuentes del documento. Cuanto más grande y oscura aparece una palabra, mayor es su frecuencia en el texto preprocesado.

In [ ]:
def generar_wordcloud(texto_limpio):
    """
    Genera y muestra una nube de palabras a partir del texto preprocesado.
    
    Args:
        texto_limpio (str): Texto ya preprocesado (sin stopwords).
    """
    wc = WordCloud(
        width=900,
        height=400,
        background_color='white',
        colormap='Blues',
        max_words=100,
        stopwords=stop_words
    ).generate(texto_limpio)
    
    plt.figure(figsize=(12, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Nube de Palabras – Documento Académico', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

generar_wordcloud(texto_limpio)

## Fase 9. Topic Modeling con LDA

**LDA** (Latent Dirichlet Allocation) es un modelo probabilístico no supervisado que descubre temas latentes en el texto. Cada tema es una distribución de palabras, y cada documento es una mezcla de temas.

In [ ]:
def topic_modeling_lda(texto_limpio, n_topics=4, n_palabras=8):
    """
    Aplica LDA para descubrir temas latentes en el documento.
    
    Args:
        texto_limpio (str): Texto preprocesado.
        n_topics (int): Número de temas a descubrir.
        n_palabras (int): Palabras por tema a mostrar.
    Returns:
        pd.DataFrame: Temas con sus palabras clave representativas.
    """
    vectorizer = TfidfVectorizer(
        max_features=500,
        stop_words=list(stop_words),
        ngram_range=(1, 1),
        min_df=1
    )
    matriz = vectorizer.fit_transform([texto_limpio])
    
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        max_iter=20
    )
    lda.fit(matriz)
    
    vocab = vectorizer.get_feature_names_out()
    temas = []
    for i, topic in enumerate(lda.components_):
        top_idx = topic.argsort()[-n_palabras:][::-1]
        palabras = [vocab[j] for j in top_idx]
        temas.append({'Tema': f'Tema {i+1}', 'Palabras clave': ', '.join(palabras)})
    
    return pd.DataFrame(temas)

print("✓ Función definida: topic_modeling_lda")

In [ ]:
# Aplicar LDA con 4 temas
df_temas = topic_modeling_lda(texto_limpio, n_topics=4, n_palabras=8)

print("Temas descubiertos por LDA:")
df_temas

In [ ]:
# Visualización de temas LDA
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, row in df_temas.iterrows():
    palabras = row['Palabras clave'].split(', ')
    y_pos = range(len(palabras))
    axes[i].barh(list(y_pos), [1]*len(palabras), color=colores[i], alpha=0.7)
    axes[i].set_yticks(list(y_pos))
    axes[i].set_yticklabels(palabras, fontsize=9)
    axes[i].invert_yaxis()
    axes[i].set_title(f'Tema {i+1}', fontsize=12, fontweight='bold', color=colores[i])
    axes[i].set_xticks([])

plt.suptitle('Temas Latentes Descubiertos por LDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Fase 10. Extracción de secciones académicas

Se utiliza **expresiones regulares** para identificar y extraer las secciones estructurales del documento: Resumen, Abstract, Introducción, Metodología, Resultados y Conclusiones.

In [ ]:
def extraer_seccion(texto, inicio, finales):
    """
    Extrae el contenido de una sección delimitada por patrones regex.
    
    Args:
        texto (str): Texto completo.
        inicio (str): Patrón regex del encabezado inicial.
        finales (list): Lista de patrones que marcan el fin de la sección.
    Returns:
        str: Contenido de la sección o 'No encontrado'.
    """
    patron = rf'(?:{inicio})\s*(.*?)(?:{"  |  ".join(finales)})'
    resultado = re.search(patron, texto, flags=re.IGNORECASE | re.DOTALL)
    if resultado:
        return resultado.group(1).strip()
    return "No encontrado"

print("✓ Función definida: extraer_seccion")

In [ ]:
# Extraer secciones principales del documento
resumen = extraer_seccion(
    texto_documento, r'\bResumen\b',
    [r'\bPalabras clave\b', r'\bAbstract\b', r'\bIntroducción\b']
)

abstract = extraer_seccion(
    texto_documento, r'\bAbstract\b',
    [r'\bKeywords\b', r'\bIntroducción\b', r'\bIntroduction\b']
)

introduccion = extraer_seccion(
    texto_documento, r'\bIntroducción\b',
    [r'\bMetodología\b', r'\bMétodo\b', r'\bResultados\b']
)

metodologia = extraer_seccion(
    texto_documento, r'\bMetodología\b|\bMétodo\b',
    [r'\bResultados\b', r'\bDiscusión\b', r'\bConclusiones\b']
)

resultados = extraer_seccion(
    texto_documento, r'\bResultados\b',
    [r'\bDiscusión\b', r'\bConclusiones\b', r'\bReferencias\b']
)

conclusiones = extraer_seccion(
    texto_documento, r'\bConclusiones\b|\bConclusión\b',
    [r'\bReferencias\b', r'\bBibliografía\b']
)

# Estado de detección
secciones = {
    'Resumen': resumen, 'Abstract': abstract,
    'Introducción': introduccion, 'Metodología': metodologia,
    'Resultados': resultados, 'Conclusiones': conclusiones
}

for nombre, contenido in secciones.items():
    estado = "✅ Detectada" if contenido != "No encontrado" else "❌ No encontrada"
    chars = len(contenido) if contenido != "No encontrado" else 0
    print(f"{estado} | {nombre} ({chars:,} caracteres)")

In [ ]:
# Vista de las secciones detectadas
for nombre, contenido in secciones.items():
    if contenido != "No encontrado":
        print(f"\n{'='*60}")
        print(f"SECCIÓN: {nombre}")
        print('='*60)
        print(contenido[:800])

## Fase 11. Extracción de metadatos adicionales

Extracción de información específica mediante expresiones regulares: años, software y métodos mencionados en el texto.

In [ ]:
# Años detectados
def extraer_anios(texto):
    return sorted(set(re.findall(r'\b(19\d{2}|20\d{2})\b', texto)))

# Software detectado
def extraer_software(texto):
    lista = ["R", "Python", "SPSS", "Stata", "SAS", "Excel",
             "Power BI", "Tableau", "NVivo", "Atlas.ti", "MATLAB"]
    return [s for s in lista if re.search(r'\b' + re.escape(s) + r'\b', texto, re.IGNORECASE)]

# Métodos detectados
def extraer_metodos(texto):
    metodos = [
        "regresión lineal", "regresión logística", "anova",
        "correlación", "clustering", "k-means", "pca",
        "árbol de decisión", "random forest", "machine learning",
        "topic modeling", "lda", "procesamiento de lenguaje natural",
        "tf-idf", "análisis de sentimiento"
    ]
    return [m for m in metodos if m.lower() in texto.lower()]

anios    = extraer_anios(texto_documento)
software = extraer_software(texto_documento)
metodos  = extraer_metodos(texto_documento)

print("Años detectados:", anios)
print("Software detectado:", software)
print("Métodos detectados:", metodos)

## Fase 12. Consolidación final de resultados

In [ ]:
def lista_a_texto(lista):
    if isinstance(lista, list):
        return ', '.join(str(x) for x in lista) if lista else 'No encontrado'
    return str(lista)

# Tabla de resumen consolidado
resumen_final = {
    'Años detectados':           lista_a_texto(anios),
    'Posibles autores':          lista_a_texto(autores[:10]),
    'Posibles instituciones':    lista_a_texto(instituciones[:10]),
    'Software detectado':        lista_a_texto(software),
    'Métodos detectados':        lista_a_texto(metodos),
    'Top keywords TF-IDF':       ', '.join(df_keywords['Término'].head(10).tolist()),
    'Temas LDA':                 str(len(df_temas)),
    'Sentimiento predominante':  df_sentimiento['Sentimiento'].mode()[0] if not df_sentimiento.empty else 'N/A',
    'Polaridad promedio':        str(round(df_sentimiento['Polaridad'].mean(), 4)) if not df_sentimiento.empty else 'N/A',
    'Resumen encontrado':        'Sí' if resumen != 'No encontrado' else 'No',
    'Abstract encontrado':       'Sí' if abstract != 'No encontrado' else 'No',
    'Introducción encontrada':   'Sí' if introduccion != 'No encontrado' else 'No',
    'Metodología encontrada':    'Sí' if metodologia != 'No encontrado' else 'No',
    'Resultados encontrados':    'Sí' if resultados != 'No encontrado' else 'No',
    'Conclusiones encontradas':  'Sí' if conclusiones != 'No encontrado' else 'No',
}

df_resumen_final = pd.DataFrame({
    'Campo': resumen_final.keys(),
    'Resultado': resumen_final.values()
})

print("RESUMEN FINAL DEL ANÁLISIS")
print("="*60)
df_resumen_final

## Fase 13. Exportación de resultados

In [ ]:
# Exportar todos los resultados a CSV

df_resumen_final.to_csv('resumen_academico_final.csv', index=False, encoding='utf-8-sig')
df_entidades_unicas.to_csv('entidades_ner.csv', index=False, encoding='utf-8-sig')
df_keywords.to_csv('keywords_tfidf.csv', index=False, encoding='utf-8-sig')
df_sentimiento.to_csv('sentimiento.csv', index=False, encoding='utf-8-sig')
df_temas.to_csv('temas_lda.csv', index=False, encoding='utf-8-sig')

print("✓ Archivos exportados:")
print("  - resumen_academico_final.csv")
print("  - entidades_ner.csv")
print("  - keywords_tfidf.csv")
print("  - sentimiento.csv")
print("  - temas_lda.csv")

In [ ]:
# Si usas Google Colab, descarga los archivos:
# from google.colab import files
# files.download('resumen_academico_final.csv')
# files.download('entidades_ner.csv')
# files.download('keywords_tfidf.csv')
# files.download('sentimiento.csv')
# files.download('temas_lda.csv')

print("Proyecto PC3 completado exitosamente.")
print("Técnicas PLN aplicadas: NER, TF-IDF, Análisis de Sentimiento, WordCloud, LDA, Extracción de secciones")